[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_01/NB_bishop_ch01_polynomial_fitting.ipynb)

# Chapter 1: Polynomial Curve Fitting

This notebook implements the polynomial curve fitting example from Bishop's *Deep Learning: Foundations and Concepts*, Chapter 1. We generate sinusoidal data with Gaussian noise, fit polynomials of varying degree, and examine overfitting and regularization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)

## 1. Generate Synthetic Data

We sample $N$ points from $t = \sin(2\pi x) + \varepsilon$, where $\varepsilon \sim \mathcal{N}(0, \sigma^2)$.

In [ ]:
np.random.seed(42)

N_train = 10
N_test = 100
sigma = 0.3

x_train = np.linspace(0, 1, N_train)
t_train = np.sin(2 * np.pi * x_train) + np.random.normal(0, sigma, N_train)

x_test = np.linspace(0, 1, N_test)
t_test = np.sin(2 * np.pi * x_test) + np.random.normal(0, sigma, N_test)

x_plot = np.linspace(0, 1, 300)
t_true = np.sin(2 * np.pi * x_plot)

print(f'Training points: {N_train}, Test points: {N_test}')

## 2. Polynomial Fitting Functions

We fit $y(x, \mathbf{w}) = \sum_{j=0}^{M} w_j x^j$ by minimizing the sum-of-squares error.

In [ ]:
def design_matrix(x, M):
    """Create the Vandermonde design matrix for polynomial degree M."""
    return np.vander(x, M + 1, increasing=True)

def fit_polynomial(x, t, M):
    """Fit polynomial of degree M using least squares (normal equations)."""
    Phi = design_matrix(x, M)
    w = np.linalg.lstsq(Phi, t, rcond=None)[0]
    return w

def predict(x, w):
    """Predict using polynomial weights."""
    M = len(w) - 1
    Phi = design_matrix(x, M)
    return Phi @ w

def rmse(predictions, targets):
    """Root mean squared error."""
    return np.sqrt(np.mean((predictions - targets) ** 2))

## 3. Fit Polynomials of Degree M = 0, 1, 3, 9

In [ ]:
degrees = [0, 1, 3, 9]
weights = {}

for M in degrees:
    w = fit_polynomial(x_train, t_train, M)
    weights[M] = w
    train_err = rmse(predict(x_train, w), t_train)
    test_err = rmse(predict(x_test, w), t_test)
    print(f'M={M:2d}  |  Train RMSE: {train_err:.4f}  |  Test RMSE: {test_err:.4f}')

## 4. Plot Polynomial Fits

Visualize how each polynomial degree fits the training data relative to the true function.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ['#1a3a6e', '#cd0000', '#228B22', '#FF8C00']

for idx, (M, ax) in enumerate(zip(degrees, axes.ravel())):
    ax.scatter(x_train, t_train, facecolors='none', edgecolors='#1a3a6e',
               s=60, zorder=5, label='Training data')
    ax.plot(x_plot, t_true, color='#228B22', linewidth=1.5, linestyle='--',
            label=r'$\sin(2\pi x)$')
    y_pred = predict(x_plot, weights[M])
    ax.plot(x_plot, y_pred, color='#cd0000', linewidth=2, label=f'M = {M}')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_title(f'Polynomial Degree M = {M}')
    ax.legend(fontsize=8)

fig.suptitle('Bishop Ch 1: Polynomial Curve Fitting', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch1_polynomial_fits')
plt.show()

## 5. Train and Test RMSE vs Polynomial Degree

In [ ]:
all_degrees = list(range(10))
train_rmse = []
test_rmse = []

for M in all_degrees:
    w = fit_polynomial(x_train, t_train, M)
    train_rmse.append(rmse(predict(x_train, w), t_train))
    test_rmse.append(rmse(predict(x_test, w), t_test))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(all_degrees, train_rmse, 'o-', color='#1a3a6e', linewidth=2,
        markersize=6, label='Training RMSE')
ax.plot(all_degrees, test_rmse, 's-', color='#cd0000', linewidth=2,
        markersize=6, label='Test RMSE')
ax.set_xlabel('Polynomial Degree M')
ax.set_ylabel('RMSE')
ax.set_title('Bishop Ch 1: RMSE vs Polynomial Degree')
ax.set_xticks(all_degrees)
ax.set_ylim(0, 1.2)
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch1_rmse_curves')
plt.show()

## 6. Regularization

We add a penalty term $\frac{\lambda}{2} \|\mathbf{w}\|^2$ to the error function. The regularized solution becomes:
$$\mathbf{w}^* = (\Phi^T \Phi + \lambda I)^{-1} \Phi^T \mathbf{t}$$

In [ ]:
def fit_polynomial_regularized(x, t, M, lam):
    """Fit polynomial with L2 regularization (ridge regression)."""
    Phi = design_matrix(x, M)
    I = np.eye(M + 1)
    w = np.linalg.solve(Phi.T @ Phi + lam * I, Phi.T @ t)
    return w

In [ ]:
M_high = 9
log_lambdas = [-np.inf, -18, -6, 0]
labels = [r'$\ln\lambda = -\infty$ (no reg.)', r'$\ln\lambda = -18$',
          r'$\ln\lambda = -6$', r'$\ln\lambda = 0$']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors_reg = ['#cd0000', '#FF8C00', '#228B22', '#1a3a6e']

for idx, (ln_lam, label, ax) in enumerate(zip(log_lambdas, labels, axes.ravel())):
    lam = np.exp(ln_lam) if ln_lam > -30 else 0.0
    w = fit_polynomial_regularized(x_train, t_train, M_high, lam)
    y_pred = predict(x_plot, w)

    ax.scatter(x_train, t_train, facecolors='none', edgecolors='#1a3a6e',
               s=60, zorder=5, label='Training data')
    ax.plot(x_plot, t_true, color='#228B22', linewidth=1.5, linestyle='--',
            label=r'$\sin(2\pi x)$')
    ax.plot(x_plot, y_pred, color=colors_reg[idx], linewidth=2, label=label)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_title(f'M = {M_high}, {label}')
    ax.legend(fontsize=7)

fig.suptitle('Bishop Ch 1: Effect of Regularization', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch1_regularization')
plt.show()

## 7. Regularization: RMSE vs ln(lambda)

In [ ]:
ln_lambdas = np.linspace(-40, 2, 50)
train_rmse_reg = []
test_rmse_reg = []

for ln_lam in ln_lambdas:
    lam = np.exp(ln_lam)
    w = fit_polynomial_regularized(x_train, t_train, M_high, lam)
    train_rmse_reg.append(rmse(predict(x_train, w), t_train))
    test_rmse_reg.append(rmse(predict(x_test, w), t_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ln_lambdas, train_rmse_reg, '-', color='#1a3a6e', linewidth=2,
        label='Training RMSE')
ax.plot(ln_lambdas, test_rmse_reg, '-', color='#cd0000', linewidth=2,
        label='Test RMSE')
ax.set_xlabel(r'$\ln \lambda$')
ax.set_ylabel('RMSE')
ax.set_title('Bishop Ch 1: Regularization Effect on RMSE (M = 9)')
ax.set_ylim(0, 1.2)
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch1_regularization')
plt.show()

## 8. Weight Magnitudes

In [ ]:
import pandas as pd

w_noreg = fit_polynomial(x_train, t_train, 9)
w_reg = fit_polynomial_regularized(x_train, t_train, 9, np.exp(-6))

df = pd.DataFrame({
    'Coefficient': [f'w_{i}' for i in range(10)],
    'No Regularization': w_noreg,
    'ln(lambda) = -6': w_reg
})
print(df.to_string(index=False, float_format='{:.4f}'.format))

## 9. Effect of Dataset Size on Overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, N in zip(axes, [15, 100]):
    np.random.seed(42)
    x_n = np.linspace(0, 1, N)
    t_n = np.sin(2 * np.pi * x_n) + np.random.normal(0, sigma, N)
    w_n = fit_polynomial(x_n, t_n, 9)
    y_pred = predict(x_plot, w_n)

    ax.scatter(x_n, t_n, facecolors='none', edgecolors='#1a3a6e',
               s=40, zorder=5, label='Data')
    ax.plot(x_plot, t_true, '--', color='#228B22', linewidth=1.5,
            label=r'$\sin(2\pi x)$')
    ax.plot(x_plot, y_pred, color='#cd0000', linewidth=2, label='M = 9 fit')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title(f'N = {N}, M = 9')
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.legend(fontsize=8)

fig.suptitle('Bishop Ch 1: Effect of Dataset Size', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 10. Summary

**Key takeaways from Bishop Chapter 1:**

- **Underfitting** (low M): the model is too simple to capture the underlying pattern.
- **Overfitting** (high M, small N): the model fits noise in the training data.
- **Regularization** controls model complexity and mitigates overfitting.
- More data naturally reduces overfitting even for complex models.
- The bias-variance trade-off is central to model selection.